# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [ ]:
# YOUR CHANGES HERE
import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are


def tsv_to_array(path):
    df = pd.read_csv(path, sep="\t", comment="#")
    # drop an 'index' column if present (common when saving with pandas)
    if 'index' in df.columns:
        df = df.drop(columns=['index'])
    return df.values.astype(float)

A = tsv_to_array("model-A.tsv")
B = tsv_to_array("model-B.tsv")
Q = tsv_to_array("cost-Q.tsv")
R = tsv_to_array("cost-R.tsv")

if R.ndim == 0:
    R = R.reshape((1, 1))

P = solve_discrete_are(A, B, Q, R)

S = R + B.T @ P @ B
K = np.linalg.solve(S, B.T @ P @ A)  
print("Computed K:", K)


Computed K: [[0.33445985 1.30445413 1.85813088]]


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [3]:
# YOUR CHANGES HERE

cols = ['x', 'v', 'a']
dfK = pd.DataFrame([K.flatten()], columns=cols)
dfK.to_csv("control-K-intended.tsv", sep="\t", index=False)
print('Saved control-K-intended.tsv')
...

Saved control-K-intended.tsv


Ellipsis

Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [ ]:
# YOUR CHANGES HERE

df_log = pd.read_csv("qc-log.tsv", sep="\t", comment="#")

# preserve or create an index column for output
if 'index' in df_log.columns:
    out_index = df_log['index'].values
else:
    out_index = np.arange(len(df_log))

# determine state columns (expect x, v, a)
expected = ['x', 'v', 'a']
if all(c in df_log.columns for c in expected):
    state_cols = expected
else:
    col_map = {c.lower(): c for c in df_log.columns}
    if all(c in col_map for c in expected):
        state_cols = [col_map[c] for c in expected]
    
X = df_log[state_cols].values  

# compute actions: u_check = -K x
u_check = - (X @ K.T).flatten()

# save results
df_out = pd.DataFrame({'index': out_index, 'u_check': u_check})
df_out.to_csv("qc-check.tsv", sep="\t", index=False)


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [5]:
# YOUR CHANGES HERE
df_out = pd.DataFrame({'index': out_index, 'u_check': u_check})
df_out.to_csv("qc-check.tsv", sep="\t", index=False)
...

Ellipsis

Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [8]:
df_log = pd.read_csv("qc-log.tsv", sep="\t")
X = df_log[['x','v','a']].astype(float).values
u_logged = df_log['u'].astype(float).values

beta = np.linalg.lstsq(X, u_logged, rcond=None)[0]
K_actual = -beta.reshape(1, -1)



Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [9]:
# YOUR CHANGES HERE
cols = ['x', 'v', 'a']
dfK_actual = pd.DataFrame([K_actual.flatten()], columns=cols)
dfK_actual.to_csv("control-K-actual.tsv", sep="\t", index=False)
print("Saved control-K-actual.tsv")
...

Saved control-K-actual.tsv


Ellipsis

Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [10]:
# YOUR CHANGES HERE
df = pd.read_csv("qc-train.tsv", sep="\t")
if 'index' in df.columns:
    df = df.drop(columns=['index'])

X = df[['x','v','a']].astype(float).values
U = df['u'].astype(float).values.reshape(-1,1)

# use t -> t+1 pairs
X_t = X[:-1]
U_t = U[:-1]
X_next = X[1:]

Z = np.hstack([X_t, U_t])            # shape (M, 4)
Theta, *_ = np.linalg.lstsq(Z, X_next, rcond=None)  # shape (4,3)

A = Theta[:3, :].T   # (3,3)
B = Theta[3:, :].T   # (3,1)


...

Ellipsis

Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [12]:
pd.DataFrame(A, columns=['x','v','a']).to_csv("model-A-new.tsv", sep="\t", index=False)
pd.DataFrame(B, columns=['b']).to_csv("model-B-new.tsv", sep="\t", index=False)
print("Saved model-A-new.tsv and model-B-new.tsv")


Saved model-A-new.tsv and model-B-new.tsv


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.